# Titanic — ML Model Deployment and Inference using FastAPI

**Google Colab notebook for the uploaded dataset `train-selected-columns.csv`**

This notebook is adapted to the exact dataset supplied with the assignment. It has **891 rows and 10 columns**:
`PassengerId`, `Survived`, `Pclass`, `Name`, `Sex`, `Age`, `SibSp`, `Parch`, `Ticket`, and `Fare`.

The dataset does **not** contain `Embarked` or `Cabin`, so this notebook does not use those features. The target is `Survived` (`0` = did not survive, `1` = survived). Missing `Age` values are handled in the model pipeline.

### Tasks covered
1. Dataset inspection and preprocessing
2. Train/test split, model training, accuracy evaluation, and `model.pkl`
3. Online inference for one held-out passenger
4. Batch inference for 20 held-out passengers → `batch_predictions.csv`
5. Streaming inference for 10 held-out passengers, one second apart
6. FastAPI `GET /health` and `POST /predict`, Pydantic validation, API tests
7. Observation report in both TXT and PDF formats

**How to use:** Upload this notebook to [Google Colab](https://colab.research.google.com/), run cells from top to bottom, and upload your provided `train-selected-columns.csv` when prompted.

## Step 1 — Install libraries and upload the provided dataset

In [ ]:
!pip -q install pandas numpy scikit-learn joblib fastapi uvicorn requests reportlab

In [ ]:
from google.colab import files
uploaded = files.upload()
csv_candidates = [name for name in uploaded if name.lower().endswith(".csv")]
if not csv_candidates:
    raise FileNotFoundError("Upload your provided CSV file: train-selected-columns.csv")
DATA_PATH = "train-selected-columns.csv" if "train-selected-columns.csv" in csv_candidates else csv_candidates[0]
print("Using dataset:", DATA_PATH)

## Step 2 — Inspect dataset details

In [ ]:
import os, time, textwrap, subprocess, sys
import numpy as np
import pandas as pd
from IPython.display import display

df = pd.read_csv(DATA_PATH)
print("Dataset file:", DATA_PATH)
print("Dataset shape (rows, columns):", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nFirst five rows:")
display(df.head())
print("\nData types and non-null counts:")
df.info()
print("\nMissing values per column:")
display(df.isnull().sum().to_frame("missing_values"))
print("\nTarget distribution (0 = did not survive, 1 = survived):")
display(df["Survived"].value_counts().sort_index().to_frame("count"))
print("\nSummary statistics:")
display(df.describe(include="all").T)

required = {"PassengerId", "Survived", "Pclass", "Sex", "Age", "SibSp", "Parch", "Fare"}
missing_required = required - set(df.columns)
if missing_required:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing_required)}")
if df["Survived"].isna().any():
    raise ValueError("The target column Survived contains missing values.")
print("\nDataset validation passed.")

## Dataset dictionary

| Column | Meaning |
|---|---|
| `PassengerId` | Passenger identifier |
| `Survived` | Target: 0 = No, 1 = Yes |
| `Pclass` | Ticket class: 1 = first, 2 = second, 3 = third |
| `Name` | Passenger name |
| `Sex` | Passenger sex |
| `Age` | Age in years; some values are missing |
| `SibSp` | Siblings/spouses aboard |
| `Parch` | Parents/children aboard |
| `Ticket` | Ticket number |
| `Fare` | Passenger fare |

**Dataset-specific note:** This supplied CSV has 10 columns and does not include `Cabin` or `Embarked`. The model uses the available passenger features `Pclass`, `Sex`, `Age`, `SibSp`, `Parch`, and `Fare`. `PassengerId`, `Name`, and `Ticket` are excluded from this baseline model.

## Step 3 — Preprocess, split, train, evaluate, and save `model.pkl`

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

TARGET = "Survived"
FEATURES = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare"]
X = df[FEATURES].copy()
y = df[TARGET].astype(int)

numeric_features = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
categorical_features = ["Sex"]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300, random_state=42, class_weight="balanced"
    ))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"Test accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

MODEL_PATH = "model.pkl"
joblib.dump(pipeline, MODEL_PATH)
print(f"Saved complete preprocessing + model pipeline to {MODEL_PATH}")

## Step 4 — Online inference (one held-out passenger)

In [ ]:
# Held-out records were not used to fit the model.
heldout = X_test.copy()
heldout["ActualSurvived"] = y_test
heldout["PassengerId"] = df.loc[X_test.index, "PassengerId"]
heldout = heldout.reset_index(drop=True)

loaded_model = joblib.load(MODEL_PATH)
one_record = heldout[FEATURES].iloc[[0]]
one_prediction = int(loaded_model.predict(one_record)[0])
print("Passenger ID:", int(heldout.loc[0, "PassengerId"]))
print("Passenger features:")
display(one_record)
print("Predicted Survived:", one_prediction, "(0 = No, 1 = Yes)")
print("Actual Survived (for local demonstration only):", int(heldout.loc[0, "ActualSurvived"]))

## Step 5 — Batch inference (20 held-out passengers)

In [ ]:
if len(heldout) < 20:
    raise ValueError("At least 20 held-out rows are required for batch inference.")
batch_records = heldout[["PassengerId"] + FEATURES].iloc[:20].copy()
batch_predictions = loaded_model.predict(batch_records[FEATURES]).astype(int)
batch_output = batch_records.copy()
batch_output["PredictedSurvived"] = batch_predictions
batch_output.to_csv("batch_predictions.csv", index=False)
display(batch_output)
print("Saved batch_predictions.csv with", len(batch_output), "predictions.")

## Step 6 — Streaming inference (10 held-out passengers, one-second delay)

In [ ]:
stream_records = heldout[["PassengerId"] + FEATURES].iloc[:10].copy()
stream_results = []
print("Starting simulated stream...")
for i, (_, record) in enumerate(stream_records.iterrows(), start=1):
    passenger_features = record[FEATURES].to_frame().T
    prediction = int(loaded_model.predict(passenger_features)[0])
    passenger_id = int(record["PassengerId"])
    print(f"Incoming record {i}/10 | PassengerId={passenger_id} | Predicted Survived={prediction}")
    stream_results.append({
        "PassengerId": passenger_id,
        "stream_record": i,
        "PredictedSurvived": prediction
    })
    if i < len(stream_records):
        time.sleep(1)
stream_output = pd.DataFrame(stream_results)
display(stream_output)
stream_output.to_csv("streaming_predictions.csv", index=False)
print("Saved streaming_predictions.csv")

## Step 7 — Create FastAPI app (`app.py`)

In [ ]:
app_code = r"""
from fastapi import FastAPI
from pydantic import BaseModel, Field
from typing import Literal, Optional
import joblib
import pandas as pd

app = FastAPI(title="Titanic Survival Prediction API", version="1.0.0")
model = joblib.load("model.pkl")

class PassengerInput(BaseModel):
    Pclass: Literal[1, 2, 3]
    Sex: Literal["female", "male"]
    Age: Optional[float] = Field(default=None, ge=0, le=120)
    SibSp: int = Field(ge=0, le=20)
    Parch: int = Field(ge=0, le=20)
    Fare: float = Field(ge=0)

@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": model is not None}

@app.post("/predict")
def predict(passenger: PassengerInput):
    # Pydantic validates the input. The model pipeline handles missing Age.
    row = pd.DataFrame([passenger.dict()])
    prediction = int(model.predict(row)[0])
    return {
        "predicted_survived": prediction,
        "label": "Survived" if prediction == 1 else "Did not survive"
    }
"""
with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code.strip() + "\n")
print("Created app.py")
print(app_code)

## Step 8 — Create API client (`client.py`)

In [ ]:
client_code = r"""
import requests

BASE_URL = "http://127.0.0.1:8000"

health_response = requests.get(f"{BASE_URL}/health", timeout=10)
print("Health:", health_response.status_code, health_response.json())

passenger = {
    "Pclass": 3,
    "Sex": "male",
    "Age": 25,
    "SibSp": 0,
    "Parch": 0,
    "Fare": 7.25
}
response = requests.post(f"{BASE_URL}/predict", json=passenger, timeout=10)
print("Valid prediction:", response.status_code, response.json())

invalid_passenger = {**passenger, "Pclass": 9}
invalid_response = requests.post(
    f"{BASE_URL}/predict", json=invalid_passenger, timeout=10
)
print("Invalid request (expected HTTP 422):", invalid_response.status_code, invalid_response.text)
"""
with open("client.py", "w", encoding="utf-8") as f:
    f.write(client_code.strip() + "\n")
print("Created client.py")
print(client_code)

## Step 9 — Create requirements and PDF observation report

In [ ]:
requirements = """pandas
numpy
scikit-learn
joblib
fastapi
uvicorn
requests
reportlab
"""
with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write(requirements)

observations = f"""TITANIC ML DEPLOYMENT — OBSERVATIONS

Dataset
- Input file: {DATA_PATH}
- Rows: {len(df)}
- Columns: {len(df.columns)}
- Features used: {", ".join(FEATURES)}
- Test split accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)

1. Online Inference vs Batch Inference
Online inference predicts one incoming record per request and returns the result immediately. Batch inference processes a group of records together and is useful for bulk processing.

2. Streaming Inference vs Batch Inference
Streaming inference processes records as they arrive, one at a time, with low latency. Batch inference processes a group of records together.

3. Why FastAPI is used for model serving
FastAPI exposes the model through HTTP endpoints, validates incoming data using Pydantic, and provides interactive API documentation at /docs.

4. Why reuse the same trained model?
Reusing the same saved pipeline keeps preprocessing and predictions consistent across online, batch, streaming, and API inference. It also avoids retraining for every request.
"""
with open("observations_report.txt", "w", encoding="utf-8") as f:
    f.write(observations)

from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors

pdf_path = "report.pdf"
doc = SimpleDocTemplate(pdf_path, pagesize=A4, rightMargin=48, leftMargin=48, topMargin=48, bottomMargin=48)
styles = getSampleStyleSheet()
story = [
    Paragraph("Titanic — ML Model Deployment and Inference using FastAPI", styles["Title"]),
    Spacer(1, 12),
    Paragraph("Dataset and model results", styles["Heading2"]),
    Paragraph(f"Dataset: {DATA_PATH}. Rows: {len(df)}; columns: {len(df.columns)}.", styles["BodyText"]),
    Paragraph(f"Features used: {', '.join(FEATURES)}.", styles["BodyText"]),
    Paragraph(f"Test accuracy: {accuracy:.4f} ({accuracy*100:.2f}%).", styles["BodyText"]),
    Spacer(1, 10),
    Paragraph("Observations", styles["Heading2"]),
]
qa = [
    ["Question", "Answer"],
    ["Online vs Batch", "Online inference predicts one record at a time. Batch inference predicts a group of records together."],
    ["Streaming vs Batch", "Streaming processes records as they arrive, one at a time. Batch processes a collected group."],
    ["Why FastAPI?", "It provides HTTP endpoints, Pydantic request validation, and automatic interactive documentation at /docs."],
    ["Why reuse one model?", "It ensures consistent preprocessing and prediction behavior and avoids retraining for each inference strategy."]
]
table = Table(qa, colWidths=[105, 360], repeatRows=1)
table.setStyle(TableStyle([
    ("BACKGROUND", (0,0), (-1,0), colors.lightgrey),
    ("GRID", (0,0), (-1,-1), 0.5, colors.grey),
    ("VALIGN", (0,0), (-1,-1), "TOP"),
    ("PADDING", (0,0), (-1,-1), 6),
]))
story.append(table)
story += [Spacer(1, 12), Paragraph("Inference outputs", styles["Heading2"]),
          Paragraph("This notebook demonstrates online prediction for one held-out passenger, batch prediction for 20 held-out passengers, and simulated streaming prediction for 10 held-out passengers with a one-second interval.", styles["BodyText"]),
          Spacer(1, 8), Paragraph("API testing", styles["Heading2"]),
          Paragraph("The generated app.py exposes GET /health and POST /predict. Test a valid request and an invalid Pclass value (expected HTTP 422) using /docs or client.py after starting the server.", styles["BodyText"])]
doc.build(story)
print("Created requirements.txt, observations_report.txt, and report.pdf")
print(observations)

## Step 10 — Start FastAPI server and test endpoints

In [ ]:
# Start FastAPI in the background in this Colab runtime.
server_log = open("uvicorn.log", "w")
server_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "app:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=server_log, stderr=subprocess.STDOUT
)
time.sleep(3)
print("Started Uvicorn. If tests fail, inspect uvicorn.log.")

In [ ]:
import requests
base_url = "http://127.0.0.1:8000"
try:
    r_health = requests.get(base_url + "/health", timeout=10)
    print("GET /health:", r_health.status_code, r_health.json())

    valid = {"Pclass": 3, "Sex": "male", "Age": 25, "SibSp": 0, "Parch": 0, "Fare": 7.25}
    r_valid = requests.post(base_url + "/predict", json=valid, timeout=10)
    print("POST /predict valid:", r_valid.status_code, r_valid.json())

    invalid = {**valid, "Pclass": 9}
    r_invalid = requests.post(base_url + "/predict", json=invalid, timeout=10)
    print("POST /predict invalid (expected 422):", r_invalid.status_code, r_invalid.text[:500])
except Exception as exc:
    print("API test error:", repr(exc))
    print("Check uvicorn.log. You can run locally with: uvicorn app:app --reload")

## Step 11 — Download generated artifacts

Run this cell after the earlier cells succeed. Save screenshots separately for your submission: dataset details, model accuracy, online inference, batch inference, streaming output, and API valid/invalid tests.

After downloading the files, include them in your public GitHub repository with a README.md. The PDF report is generated as `report.pdf`.

In [ ]:
from google.colab import files
artifacts = [
    "model.pkl",
    "batch_predictions.csv",
    "streaming_predictions.csv",
    "app.py",
    "client.py",
    "requirements.txt",
    "observations_report.txt",
    "report.pdf"
]
for artifact in artifacts:
    if os.path.exists(artifact):
        files.download(artifact)
    else:
        print("Not found:", artifact)